In [ ]:
# --- 0. IMPORTS ---
#This block same for all locations
#Change only dataset name
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GRU, Dropout, Conv1D, Input, Bidirectional, Layer
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
import warnings
import os

# --- 1. SETUP ---
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

try:
    strategy = tf.distribute.MirroredStrategy()
    GLOBAL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync
except Exception:
    strategy = tf.distribute.get_strategy()
    GLOBAL_BATCH_SIZE = 64

# --- 2. MODEL & ATTENTION (Copied from main script) ---
class Attention(Layer):
    def __init__(self,**kwargs): super(Attention,self).__init__(**kwargs)
    def build(self, input_shape):
        self.W=self.add_weight(name='att_W',shape=(input_shape[-1],1),initializer='glorot_uniform',trainable=True)
        self.b=self.add_weight(name='att_b',shape=(input_shape[1],1),initializer='zeros',trainable=True); super(Attention,self).build(input_shape)
    def call(self, x): e=tf.tanh(tf.tensordot(x,self.W,axes=1)+self.b); a=tf.nn.softmax(e,axis=1); o=x*a; return tf.reduce_sum(o,axis=1)

def create_tcn_bigru_model(n_timesteps, n_features):
    reg = l2(1e-4)
    i=Input(shape=(n_timesteps,n_features))
    t=Conv1D(128, 3, activation='relu', padding='causal', dilation_rate=1, kernel_regularizer=reg)(i); t=Dropout(.2)(t)
    t=Conv1D(128, 3, activation='relu', padding='causal', dilation_rate=2, kernel_regularizer=reg)(t); t=Dropout(.2)(t)
    t=Conv1D(128, 3, activation='relu', padding='causal', dilation_rate=4, kernel_regularizer=reg)(t); t=Dropout(.2)(t)
    bg=Bidirectional(GRU(192,activation='tanh',return_sequences=True, kernel_regularizer=reg))(t)
    att=Attention()(bg); p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

# --- 3. MODIFIED COMPILE FUNCTION ---
def compile_keras_model(model, prob_weight, reg_weight):
    """ This function now accepts weights as arguments. """
    model.compile(optimizer=Adam(learning_rate=.0002),
                  loss={'prob_output':BinaryCrossentropy(), 'reg_output':'mean_squared_error'}, # Use standard MSE
                  loss_weights={'prob_output': prob_weight,'reg_output': reg_weight})
    return model

# --- 4. DATA PREPARATION (Lightweight version) ---

DATASET_NAME = "silchar" # Make sure this file is available
try:
    df=pd.read_csv(f'{DATASET_NAME}.csv')
    print(f"  Successfully loaded '{DATASET_NAME}.csv'")
except FileNotFoundError:
    print(f"  Error: '{DATASET_NAME}.csv' not found. Trying fallback...")
    try:
        df=pd.read_csv(f'/kaggle/input/dataset2/{DATASET_NAME}.csv')
        print(f"  Successfully loaded from fallback path.")
    except FileNotFoundError:
        print(f"  Error: Fallback path also failed. Exiting.")
        exit()

# Pre-processing
df['Date']=pd.to_datetime({'year':df['YEAR'],'month':df['MN'],'day':df['DT']}); df.set_index('Date',inplace=True)
num_cols=['RF','MSLP','DBT','RH','ONI','DMI']; [df[c].apply(pd.to_numeric,errors='coerce') for c in num_cols]
daily=df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in num_cols}); daily.index=pd.to_datetime(daily.index); daily.dropna(subset=num_cols,inplace=True)
daily['RF_SSA']=daily['RF'].rolling(7,center=True).mean().fillna(daily['RF']); daily['RF_7m']=daily['RF'].rolling(7).mean().shift(1); daily['RF_7s']=daily['RF'].rolling(7).std().shift(1); daily['sin_d']=np.sin(2*np.pi*daily.index.dayofyear/365.25); daily['cos_d']=np.cos(2*np.pi*daily.index.dayofyear/365.25)
daily['RF_l1']=daily['RF'].shift(1); daily['DBT_l1']=daily['DBT'].shift(1); daily['RH_MSLP_I']=(daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan); daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True); daily['RF']=pd.to_numeric(daily['RF'],errors='coerce').fillna(0)
HORIZON=3; LOOK_BACK=7; FEATURES=['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']; FEATURES=[f for f in FEATURES if f in daily.columns]; N_FEATS=len(FEATURES);

def create_ts_feats(data,feats,target,lb,hz):
    X,Y=[],[]; ts=data[target].shift(-hz); ts=pd.to_numeric(ts,errors='coerce'); vfd=data[feats].dropna(); max_idx=len(vfd)-lb-hz+1; [ (X.append(fw),Y.append(ts.iloc[i+lb-1])) for i in range(max_idx) if (i+lb-1)<len(ts) and not pd.isna(ts.iloc[i+lb-1]) and (fw:=vfd.iloc[i:i+lb].values).shape[0]==lb ]; X=np.array(X).astype(np.float32); Y=np.array(Y).astype(np.float32); return X,Y

X_seq_o, Y_o = create_ts_feats(daily, FEATURES, 'RF', LOOK_BACK, HORIZON); v_idx=~np.isnan(Y_o); X_seq_o=X_seq_o[v_idx]; Y_o=Y_o[v_idx]
Y_o = np.log1p(Y_o) # <--- Apply log-transform

# --- Create Splits (Identical to main script's logic) ---
sp_idx=int(X_seq_o.shape[0]*.8)
X_tr_v_o, X_ts_o = X_seq_o[:sp_idx], X_seq_o[sp_idx:]
Y_tr_v_r, Y_ts_r = Y_o[:sp_idx], Y_o[sp_idx:]
Y_tr_v_p, Y_ts_p = (np.expm1(Y_tr_v_r)>0).astype(int), (np.expm1(Y_ts_r)>0).astype(int)

# Scale sequential features
sc=StandardScaler()
X_tr_v_f_o=X_tr_v_o.reshape(X_tr_v_o.shape[0],-1)
X_tr_v_f_s=sc.fit_transform(np.nan_to_num(X_tr_v_f_o))
X_tr_v_s=X_tr_v_f_s.reshape(X_tr_v_o.shape).astype(np.float32)

# Create the specific 64% / 16% split for this test
(X_tr_s, X_v_s,
 Y_tr_r, Y_v_r,
 Y_tr_p, Y_v_p) = train_test_split(
     X_tr_v_s, Y_tr_v_r, Y_tr_v_p, 
     test_size=0.2, random_state=GLOBAL_SEED # Use 0.2 of the 80% = 16%
)

print(f"Sensitivity Analysis Data:")
print(f"  Train (Seq): {X_tr_s.shape} | Val (Seq): {X_v_s.shape}")

# --- 5. THE SENSITIVITY ANALYSIS ---

# Define all the weight combinations you want to test
# [Prob, Reg]
weights_to_test = [
    [0.2, 1.8],   # Give regression more weight
    [0.5, 1.5],   #
    [1.0, 1.0],   # The balanced weight (our hypothesis)
    [1.5, 0.5],   #
    [1.8, 0.2]    # Give probability more weight
]

analysis_results = []
es = EarlyStopping(monitor='val_reg_output_loss', patience=20, restore_best_weights=True, mode='min')

for prob_w, reg_w in weights_to_test:
    print(f"\n--- Testing Weights: Prob={prob_w}, Reg={reg_w} ---")
    
    # Clear session and create model
    tf.keras.backend.clear_session()
    with strategy.scope():
        model = create_tcn_bigru_model(LOOK_BACK, N_FEATS)
        model = compile_keras_model(model, prob_w, reg_w)
    
    # Train the model
    history = model.fit(
        X_tr_s.astype(np.float32),
        {'prob_output': Y_tr_p.astype(np.float32), 'reg_output': Y_tr_r.astype(np.float32)},
        epochs=150,
        batch_size=GLOBAL_BATCH_SIZE,
        validation_data=(
            X_v_s.astype(np.float32),
            {'prob_output': Y_v_p.astype(np.float32), 'reg_output': Y_v_r.astype(np.float32)}
        ),
        callbacks=[es],
        verbose=1 # Show epochs for this analysis
    )
    
    # Get the best validation loss (MSE in log-space)
    best_val_log_mse = min(history.history['val_reg_output_loss'])
    
    # For a fair comparison, get standard RMSE in millimeters (mm)
    _, preds_r_log = model.predict(X_v_s.astype(np.float32), verbose=0)
    
    # Convert predictions and true values back to mm
    preds_r_mm = np.expm1(np.maximum(0, preds_r_log.flatten()))
    Y_v_r_mm = np.expm1(Y_v_r)
    
    val_rmse_mm = np.sqrt(mean_squared_error(Y_v_r_mm, preds_r_mm))
    
    print(f"  Best val_log_mse: {best_val_log_mse:.4f}")
    print(f"  Standard val_rmse_mm: {val_rmse_mm:.4f}")
    
    analysis_results.append({
        "prob_weight": prob_w,
        "reg_weight": reg_w,
        "val_log_mse": best_val_log_mse,
        "val_rmse_mm": val_rmse_mm
    })

# --- 6. PRINT THE FINAL JUSTIFICATION TABLE ---
print("\n\n" + "="*50)
print("    LOSS WEIGHT SENSITIVITY ANALYSIS RESULTS    ")
print("="*50 + "\n")
results_df = pd.DataFrame(analysis_results).sort_values(by="val_rmse_mm")
print(results_df.to_markdown(index=False, floatfmt=".4f"))

print("\n\n--- Conclusion ---")
best_choice = results_df.iloc[0]
print(f"The best performance (lowest RMSE in millimeters) was {best_choice['val_rmse_mm']:.4f} using weights Prob={best_choice['prob_weight']} and Reg={best_choice['reg_weight']}.")
print("You can use this table to justify this choice in your paper.")

In [ ]:
# ---
# --- CELL 1: ENVIRONMENT FIX (RUN THIS CELL ONCE, THEN RESTART KERNEL) ---
# ---
!pip install protobuf==3.20.3
!pip install optuna-integration[tfkeras]

In [ ]:
!pip install optuna-integration[tfkeras]

In [ ]:
# --- 0. IMPORTS ---
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GRU, LSTM, Dropout, Conv1D, Input, Bidirectional, Layer
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.integration import TFKerasPruningCallback
import warnings
import os

# --- 1. SCRIPT-WIDE SETTINGS ---
GLOBAL_SEED = 42
N_TRIALS_KERAS = 15 # <-- Set how many trials to run (e.g., 15-25)
TUNING_SUBSET_FRACTION = 0.75 # Use 75% of data for tuning (faster)

# Suppress warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)

# Set seeds
np.random.seed(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# --- 2. SETUP MULTI-GPU STRATEGY ---
try:
    strategy = tf.distribute.MirroredStrategy()
    print(f'  Multi-GPU strategy active. Number of devices: {strategy.num_replicas_in_sync}')
    GLOBAL_BATCH_SIZE = 64 * strategy.num_replicas_in_sync
    print(f'Global Batch Size: {GLOBAL_BATCH_SIZE}')
except Exception as e:
    print(f"⚠️ Could not initialize MirroredStrategy: {e}. Falling back to default strategy.")
    strategy = tf.distribute.get_strategy()
    GLOBAL_BATCH_SIZE = 64

# --- 3. CUSTOM LOSS & ATTENTION LAYER ---
class Attention(Layer):
    def __init__(self,**kwargs): super(Attention,self).__init__(**kwargs)
    def build(self, input_shape):
        self.W=self.add_weight(name='att_W',shape=(input_shape[-1],1),initializer='glorot_uniform',trainable=True)
        self.b=self.add_weight(name='att_b',shape=(input_shape[1],1),initializer='zeros',trainable=True); super(Attention,self).build(input_shape)
    def call(self, x): e=tf.tanh(tf.tensordot(x,self.W,axes=1)+self.b); a=tf.nn.softmax(e,axis=1); o=x*a; return tf.reduce_sum(o,axis=1)

# --- 4. MODEL DEFINITIONS (MODIFIED FOR OPTUNA) ---
def create_tcn_bigru_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='causal', dilation_rate=1, kernel_regularizer=reg)(i)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='causal', dilation_rate=2, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='causal', dilation_rate=4, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    bg=Bidirectional(GRU(params['gru_units'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(t)
    att=Attention()(bg)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def create_bilstm_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    l=Bidirectional(LSTM(params['lstm_units_1'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(i)
    l=Dropout(params['dropout_rate'])(l)
    l=Bidirectional(LSTM(params['lstm_units_2'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(l)
    l=Dropout(params['dropout_rate'])(l)
    att=Attention()(l)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def compile_keras_model(model, learning_rate, prob_weight=0.2, reg_weight=1.8):
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss={'prob_output':BinaryCrossentropy(), 'reg_output':'mean_squared_error'},
                  loss_weights={'prob_output': prob_weight,'reg_output': reg_weight})
    return model

# --- 5. DATA PREPARATION (GLOBAL) ---
DATASET_NAME = "silchar"
try:
    df=pd.read_csv(f'{DATASET_NAME}.csv')
    print(f"  Successfully loaded '{DATASET_NAME}.csv'")
except FileNotFoundError:
    print(f"  Error: '{DATASET_NAME}.csv' not found. Trying fallback...")
    try:
        df=pd.read_csv(f'/kaggle/input/dataset2/{DATASET_NAME}.csv')
        print(f"  Successfully loaded from fallback path.")
    except FileNotFoundError:
        print(f"  Error: Fallback path also failed. Exiting.")
        exit()

# Pre-processing
df['Date']=pd.to_datetime({'year':df['YEAR'],'month':df['MN'],'day':df['DT']}); df.set_index('Date',inplace=True)
num_cols=['RF','MSLP','DBT','RH','ONI','DMI']; [df[c].apply(pd.to_numeric,errors='coerce') for c in num_cols]
daily=df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in num_cols}); daily.index=pd.to_datetime(daily.index); daily.dropna(subset=num_cols,inplace=True)
daily['RF_SSA']=daily['RF'].rolling(7,center=True).mean().fillna(daily['RF']); daily['RF_7m']=daily['RF'].rolling(7).mean().shift(1); daily['RF_7s']=daily['RF'].rolling(7).std().shift(1); daily['sin_d']=np.sin(2*np.pi*daily.index.dayofyear/365.25); daily['cos_d']=np.cos(2*np.pi*daily.index.dayofyear/365.25)
daily['RF_l1']=daily['RF'].shift(1); daily['DBT_l1']=daily['DBT'].shift(1); daily['RH_MSLP_I']=(daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan); daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True); daily['RF']=pd.to_numeric(daily['RF'],errors='coerce').fillna(0)
HORIZON=3; LOOK_BACK=7; FEATURES=['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']; FEATURES=[f for f in FEATURES if f in daily.columns]; N_FEATS=len(FEATURES); print(f"Features ({N_FEATS}): {FEATURES}")

def create_ts_feats(data,feats,target,lb,hz):
    X,Y=[],[]; ts=data[target].shift(-hz); ts=pd.to_numeric(ts,errors='coerce'); vfd=data[feats].dropna(); max_idx=len(vfd)-lb-hz+1; [ (X.append(fw),Y.append(ts.iloc[i+lb-1])) for i in range(max_idx) if (i+lb-1)<len(ts) and not pd.isna(ts.iloc[i+lb-1]) and (fw:=vfd.iloc[i:i+lb].values).shape[0]==lb ]; X=np.array(X).astype(np.float32); Y=np.array(Y).astype(np.float32); print(f"X:{X.shape}, Y:{Y.shape}"); return X,Y

X_seq_o, Y_o = create_ts_feats(daily, FEATURES, 'RF', LOOK_BACK, HORIZON); v_idx=~np.isnan(Y_o); X_seq_o=X_seq_o[v_idx]; Y_o=Y_o[v_idx]
Y_o = np.log1p(Y_o)
print(f"Total usable samples: {X_seq_o.shape[0]}")

# --- 6. CREATE DATA SPLITS ---
sp_idx=int(X_seq_o.shape[0]*.8)
X_tr_v_o, X_ts_o = X_seq_o[:sp_idx], X_seq_o[sp_idx:]
Y_tr_v_r, Y_ts_r = Y_o[:sp_idx], Y_o[sp_idx:]
Y_tr_v_p, Y_ts_p = (np.expm1(Y_tr_v_r)>0).astype(int), (np.expm1(Y_ts_r)>0).astype(int)

# Scale sequential features
sc=StandardScaler()
X_tr_v_f_s=sc.fit_transform(np.nan_to_num(X_tr_v_o.reshape(X_tr_v_o.shape[0],-1)))
X_tr_v_s=X_tr_v_f_s.reshape(X_tr_v_o.shape).astype(np.float32)

# Create final Train/Val split for tuning
(X_tr_s, X_v_s,
 Y_tr_r, Y_v_r,
 Y_tr_p, Y_v_p) = train_test_split(
    X_tr_v_s, Y_tr_v_r, Y_tr_v_p, 
    test_size=0.2, random_state=GLOBAL_SEED
)

# Create smaller subset for Keras tuning
print(f"Creating tuning subset ({TUNING_SUBSET_FRACTION*100}%) for Keras...")
subset_size = int(len(X_tr_s) * TUNING_SUBSET_FRACTION)
indices = np.random.choice(len(X_tr_s), subset_size, replace=False)

X_tr_s_tune = X_tr_s[indices]
Y_tr_r_tune = Y_tr_r[indices]
Y_tr_p_tune = Y_tr_p[indices]
Y_tr_targets_tune = {'prob_output': Y_tr_p_tune, 'reg_output': Y_tr_r_tune}
Y_v_targets = {'prob_output': Y_v_p, 'reg_output': Y_v_r}

print(f"  Train (Seq): {X_tr_s.shape} | Val (Seq): {X_v_s.shape}")
print(f"  Keras Tune (Seq): {X_tr_s_tune.shape}")


# --- 7. TUNE L0 KERAS MODELS ---
print("\n  [GPU] Keras Model Tuning...")
es_callback = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# --- TCN-BiGRU Tuning ---
def obj_tcn(trial):
    tf.keras.backend.clear_session()
    params = {
        'conv_filters': trial.suggest_categorical('conv_filters', [64, 128, 192]),
        'gru_units': trial.suggest_categorical('gru_units', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.4),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    with strategy.scope():
        model = create_tcn_bigru_model(LOOK_BACK, N_FEATS, params)
        model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100, 
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # ---
    # --- FIX: Return only the first value (total_loss) ---
    # ---
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_tcn = optuna.create_study(direction='minimize', 
                              sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                              pruner=optuna.pruners.MedianPruner())
study_tcn.optimize(obj_tcn, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_tcn = study_tcn.best_params


# --- BiLSTM Tuning ---
def obj_lstm(trial):
    tf.keras.backend.clear_session()
    params = {
        'lstm_units_1': trial.suggest_categorical('lstm_units_1', [96, 128, 192]),
        'lstm_units_2': trial.suggest_categorical('lstm_units_2', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    with strategy.scope():
        model = create_bilstm_model(LOOK_BACK, N_FEATS, params)
        model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100,
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # ---
    # --- FIX: Return only the first value (total_loss) ---
    # ---
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_lstm = optuna.create_study(direction='minimize', 
                               sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                               pruner=optuna.pruners.MedianPruner())
study_lstm.optimize(obj_lstm, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_lstm = study_lstm.best_params


# --- 8. PRINTABLE RESULTS ---
print("\n" + "="*50)
print("    KERAS TUNING COMPLETE - BEST PARAMETERS")
print("="*50 + "\n")

print(f"  Best TCN-BiGRU (Val Loss: {study_tcn.best_value:.4f})")
print(f"bp_tcn_global = {bp_tcn}")

print("\n" + "-"*50 + "\n")

print(f"  Best BiLSTM (Val Loss: {study_lstm.best_value:.4f})")
print(f"bp_lstm_global = {bp_lstm}")

print("\n" + "="*50)
print("Copy the dictionaries above and paste them")
print("into your main script (Section 4).")
print("="*50)

In [ ]:
# --- 0. IMPORTS ---
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GRU, LSTM, Dropout, Conv1D, Input, Bidirectional, Layer
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.integration import TFKerasPruningCallback
import warnings
import os

# --- 1. SCRIPT-WIDE SETTINGS ---
GLOBAL_SEED = 42
N_TRIALS_KERAS = 15 # <-- Set how many trials to run (e.g., 15-25)
TUNING_SUBSET_FRACTION = 0.75 # Use 75% of data for tuning (faster)

# Suppress warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)

# Set seeds
np.random.seed(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# --- 2. SETUP STRATEGY (SINGLE GPU / CPU) ---
print("Running on single device (GPU/CPU). Multi-GPU strategy removed.")
# Set a fixed batch size. 128 is a good default.
GLOBAL_BATCH_SIZE = 128 
print(f'Global Batch Size: {GLOBAL_BATCH_SIZE}')


# --- 3. CUSTOM LOSS & ATTENTION LAYER ---
class Attention(Layer):
    def __init__(self,**kwargs): super(Attention,self).__init__(**kwargs)
    def build(self, input_shape):
        self.W=self.add_weight(name='att_W',shape=(input_shape[-1],1),initializer='glorot_uniform',trainable=True)
        self.b=self.add_weight(name='att_b',shape=(input_shape[1],1),initializer='zeros',trainable=True); super(Attention,self).build(input_shape)
    def call(self, x): e=tf.tanh(tf.tensordot(x,self.W,axes=1)+self.b); a=tf.nn.softmax(e,axis=1); o=x*a; return tf.reduce_sum(o,axis=1)

# --- 4. MODEL DEFINITIONS (MODIFIED FOR OPTUNA) ---
def create_tcn_bigru_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    
    # ---
    # --- FIX: Changed 'causal' to 'same' to avoid CUDNN bug ---
    # ---
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=1, kernel_regularizer=reg)(i)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=2, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=4, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    # --- End of Fix ---

    bg=Bidirectional(GRU(params['gru_units'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(t)
    att=Attention()(bg)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def create_bilstm_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    l=Bidirectional(LSTM(params['lstm_units_1'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(i)
    l=Dropout(params['dropout_rate'])(l)
    l=Bidirectional(LSTM(params['lstm_units_2'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(l)
    l=Dropout(params['dropout_rate'])(l)
    att=Attention()(l)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def compile_keras_model(model, learning_rate, prob_weight=0.2, reg_weight=1.8):
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss={'prob_output':BinaryCrossentropy(), 'reg_output':'mean_squared_error'},
                  loss_weights={'prob_output': prob_weight,'reg_output': reg_weight})
    return model

# --- 5. DATA PREPARATION (GLOBAL) ---
DATASET_NAME = "shillong" # <-- *** EDITED FOR SHILLONG ***
try:
    df=pd.read_csv(f'{DATASET_NAME}.csv')
    print(f"  Successfully loaded '{DATASET_NAME}.csv'")
except FileNotFoundError:
    print(f"  Error: '{DATASET_NAME}.csv' not found. Trying fallback...")
    try:
        df=pd.read_csv(f'/kaggle/input/dataset2/{DATASET_NAME}.csv')
        print(f"  Successfully loaded from fallback path.")
    except FileNotFoundError:
        print(f"  Error: Fallback path also failed. Exiting.")
        exit()

# Pre-processing
df['Date']=pd.to_datetime({'year':df['YEAR'],'month':df['MN'],'day':df['DT']}); df.set_index('Date',inplace=True)
num_cols=['RF','MSLP','DBT','RH','ONI','DMI']; [df[c].apply(pd.to_numeric,errors='coerce') for c in num_cols]
daily=df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in num_cols}); daily.index=pd.to_datetime(daily.index); daily.dropna(subset=num_cols,inplace=True)
daily['RF_SSA']=daily['RF'].rolling(7,center=True).mean().fillna(daily['RF']); daily['RF_7m']=daily['RF'].rolling(7).mean().shift(1); daily['RF_7s']=daily['RF'].rolling(7).std().shift(1); daily['sin_d']=np.sin(2*np.pi*daily.index.dayofyear/365.25); daily['cos_d']=np.cos(2*np.pi*daily.index.dayofyear/365.25)
daily['RF_l1']=daily['RF'].shift(1); daily['DBT_l1']=daily['DBT'].shift(1); daily['RH_MSLP_I']=(daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan); daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True); daily['RF']=pd.to_numeric(daily['RF'],errors='coerce').fillna(0)
HORIZON=3; LOOK_BACK=7; FEATURES=['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']; FEATURES=[f for f in FEATURES if f in daily.columns]; N_FEATS=len(FEATURES); print(f"Features ({N_FEATS}): {FEATURES}")

def create_ts_feats(data,feats,target,lb,hz):
    X,Y=[],[]; ts=data[target].shift(-hz); ts=pd.to_numeric(ts,errors='coerce'); vfd=data[feats].dropna(); max_idx=len(vfd)-lb-hz+1; [ (X.append(fw),Y.append(ts.iloc[i+lb-1])) for i in range(max_idx) if (i+lb-1)<len(ts) and not pd.isna(ts.iloc[i+lb-1]) and (fw:=vfd.iloc[i:i+lb].values).shape[0]==lb ]; X=np.array(X).astype(np.float32); Y=np.array(Y).astype(np.float32); print(f"X:{X.shape}, Y:{Y.shape}"); return X,Y

X_seq_o, Y_o = create_ts_feats(daily, FEATURES, 'RF', LOOK_BACK, HORIZON); v_idx=~np.isnan(Y_o); X_seq_o=X_seq_o[v_idx]; Y_o=Y_o[v_idx]
Y_o = np.log1p(Y_o)
print(f"Total usable samples: {X_seq_o.shape[0]}")

# --- 6. CREATE DATA SPLITS ---
sp_idx=int(X_seq_o.shape[0]*.8)
X_tr_v_o, X_ts_o = X_seq_o[:sp_idx], X_seq_o[sp_idx:]
Y_tr_v_r, Y_ts_r = Y_o[:sp_idx], Y_o[sp_idx:]
Y_tr_v_p, Y_ts_p = (np.expm1(Y_tr_v_r)>0).astype(int), (np.expm1(Y_ts_r)>0).astype(int)

# Scale sequential features
sc=StandardScaler()
X_tr_v_f_s=sc.fit_transform(np.nan_to_num(X_tr_v_o.reshape(X_tr_v_o.shape[0],-1)))
X_tr_v_s=X_tr_v_f_s.reshape(X_tr_v_o.shape).astype(np.float32)

# Create final Train/Val split for tuning
(X_tr_s, X_v_s,
 Y_tr_r, Y_v_r,
 Y_tr_p, Y_v_p) = train_test_split(
     X_tr_v_s, Y_tr_v_r, Y_tr_v_p, 
     test_size=0.2, random_state=GLOBAL_SEED
)

# Create smaller subset for Keras tuning
print(f"Creating tuning subset ({TUNING_SUBSET_FRACTION*100}%) for Keras...")
subset_size = int(len(X_tr_s) * TUNING_SUBSET_FRACTION)
indices = np.random.choice(len(X_tr_s), subset_size, replace=False)

X_tr_s_tune = X_tr_s[indices]
Y_tr_r_tune = Y_tr_r[indices]
Y_tr_p_tune = Y_tr_p[indices]
Y_tr_targets_tune = {'prob_output': Y_tr_p_tune, 'reg_output': Y_tr_r_tune}
Y_v_targets = {'prob_output': Y_v_p, 'reg_output': Y_v_r}

print(f"  Train (Seq): {X_tr_s.shape} | Val (Seq): {X_v_s.shape}")
print(f"  Keras Tune (Seq): {X_tr_s_tune.shape}")


# --- 7. TUNE L0 KERAS MODELS ---
print("\n  [GPU] Keras Model Tuning...")
es_callback = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# --- TCN-BiGRU Tuning ---
def obj_tcn(trial):
    tf.keras.backend.clear_session()
    params = {
        'conv_filters': trial.suggest_categorical('conv_filters', [64, 128, 192]),
        'gru_units': trial.suggest_categorical('gru_units', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.4),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    
    # Removed strategy.scope()
    model = create_tcn_bigru_model(LOOK_BACK, N_FEATS, params)
    model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100, 
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # Return only the first value (total_loss)
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_tcn = optuna.create_study(direction='minimize', 
                                sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                                pruner=optuna.pruners.MedianPruner())
study_tcn.optimize(obj_tcn, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_tcn = study_tcn.best_params


# --- BiLSTM Tuning ---
def obj_lstm(trial):
    tf.keras.backend.clear_session()
    params = {
        'lstm_units_1': trial.suggest_categorical('lstm_units_1', [96, 128, 192]),
        'lstm_units_2': trial.suggest_categorical('lstm_units_2', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    
    # Removed strategy.scope()
    model = create_bilstm_model(LOOK_BACK, N_FEATS, params)
    model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100,
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # Return only the first value (total_loss)
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_lstm = optuna.create_study(direction='minimize', 
                                 sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                                 pruner=optuna.pruners.MedianPruner())
study_lstm.optimize(obj_lstm, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_lstm = study_lstm.best_params


# --- 8. PRINTABLE RESULTS ---
print("\n" + "="*50)
print("     KERAS TUNING COMPLETE - BEST PARAMETERS")
print("="*50 + "\n")

print(f"  Best TCN-BiGRU (Val Loss: {study_tcn.best_value:.4f})")
print(f"bp_tcn_global = {bp_tcn}")

print("\n" + "-"*50 + "\n")

print(f"  Best BiLSTM (Val Loss: {study_lstm.best_value:.4f})")
print(f"bp_lstm_global = {bp_lstm}")

print(f"\n" + "="*50)
print("Copy the dictionaries above and paste them")
print("into your main script (Section 4).")
print("="*50)

In [ ]:
# --- 0. IMPORTS ---
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GRU, LSTM, Dropout, Conv1D, Input, Bidirectional, Layer
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.integration import TFKerasPruningCallback
import warnings
import os

# --- 1. SCRIPT-WIDE SETTINGS ---
GLOBAL_SEED = 42
N_TRIALS_KERAS = 15 # <-- Set how many trials to run (e.g., 15-25)
TUNING_SUBSET_FRACTION = 0.75 # Use 75% of data for tuning (faster)

# Suppress warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)

# Set seeds
np.random.seed(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# --- 2. SETUP STRATEGY (SINGLE GPU / CPU) ---
print("Running on single device (GPU/CPU). Multi-GPU strategy removed.")
# Set a fixed batch size. 128 is a good default.
GLOBAL_BATCH_SIZE = 128 
print(f'Global Batch Size: {GLOBAL_BATCH_SIZE}')


# --- 3. CUSTOM LOSS & ATTENTION LAYER ---
class Attention(Layer):
    def __init__(self,**kwargs): super(Attention,self).__init__(**kwargs)
    def build(self, input_shape):
        self.W=self.add_weight(name='att_W',shape=(input_shape[-1],1),initializer='glorot_uniform',trainable=True)
        self.b=self.add_weight(name='att_b',shape=(input_shape[1],1),initializer='zeros',trainable=True); super(Attention,self).build(input_shape)
    def call(self, x): e=tf.tanh(tf.tensordot(x,self.W,axes=1)+self.b); a=tf.nn.softmax(e,axis=1); o=x*a; return tf.reduce_sum(o,axis=1)

# --- 4. MODEL DEFINITIONS (MODIFIED FOR OPTUNA) ---
def create_tcn_bigru_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    
    # ---
    # --- FIX: Changed 'causal' to 'same' to avoid CUDNN bug ---
    # ---
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=1, kernel_regularizer=reg)(i)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=2, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=4, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    # --- End of Fix ---

    bg=Bidirectional(GRU(params['gru_units'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(t)
    att=Attention()(bg)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def create_bilstm_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    l=Bidirectional(LSTM(params['lstm_units_1'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(i)
    l=Dropout(params['dropout_rate'])(l)
    l=Bidirectional(LSTM(params['lstm_units_2'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(l)
    l=Dropout(params['dropout_rate'])(l)
    att=Attention()(l)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def compile_keras_model(model, learning_rate, prob_weight=0.2, reg_weight=1.8):
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss={'prob_output':BinaryCrossentropy(), 'reg_output':'mean_squared_error'},
                  loss_weights={'prob_output': prob_weight,'reg_output': reg_weight})
    return model

# --- 5. DATA PREPARATION (GLOBAL) ---
DATASET_NAME = "imphal" 
try:
    df=pd.read_csv(f'{DATASET_NAME}.csv')
    print(f"  Successfully loaded '{DATASET_NAME}.csv'")
except FileNotFoundError:
    print(f"  Error: '{DATASET_NAME}.csv' not found. Trying fallback...")
    try:
        df=pd.read_csv(f'/kaggle/input/dataset2/{DATASET_NAME}.csv')
        print(f"  Successfully loaded from fallback path.")
    except FileNotFoundError:
        print(f"  Error: Fallback path also failed. Exiting.")
        exit()

# Pre-processing
df['Date']=pd.to_datetime({'year':df['YEAR'],'month':df['MN'],'day':df['DT']}); df.set_index('Date',inplace=True)
num_cols=['RF','MSLP','DBT','RH','ONI','DMI']; [df[c].apply(pd.to_numeric,errors='coerce') for c in num_cols]
daily=df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in num_cols}); daily.index=pd.to_datetime(daily.index); daily.dropna(subset=num_cols,inplace=True)
daily['RF_SSA']=daily['RF'].rolling(7,center=True).mean().fillna(daily['RF']); daily['RF_7m']=daily['RF'].rolling(7).mean().shift(1); daily['RF_7s']=daily['RF'].rolling(7).std().shift(1); daily['sin_d']=np.sin(2*np.pi*daily.index.dayofyear/365.25); daily['cos_d']=np.cos(2*np.pi*daily.index.dayofyear/365.25)
daily['RF_l1']=daily['RF'].shift(1); daily['DBT_l1']=daily['DBT'].shift(1); daily['RH_MSLP_I']=(daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan); daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True); daily['RF']=pd.to_numeric(daily['RF'],errors='coerce').fillna(0)
HORIZON=3; LOOK_BACK=7; FEATURES=['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']; FEATURES=[f for f in FEATURES if f in daily.columns]; N_FEATS=len(FEATURES); print(f"Features ({N_FEATS}): {FEATURES}")

def create_ts_feats(data,feats,target,lb,hz):
    X,Y=[],[]; ts=data[target].shift(-hz); ts=pd.to_numeric(ts,errors='coerce'); vfd=data[feats].dropna(); max_idx=len(vfd)-lb-hz+1; [ (X.append(fw),Y.append(ts.iloc[i+lb-1])) for i in range(max_idx) if (i+lb-1)<len(ts) and not pd.isna(ts.iloc[i+lb-1]) and (fw:=vfd.iloc[i:i+lb].values).shape[0]==lb ]; X=np.array(X).astype(np.float32); Y=np.array(Y).astype(np.float32); print(f"X:{X.shape}, Y:{Y.shape}"); return X,Y

X_seq_o, Y_o = create_ts_feats(daily, FEATURES, 'RF', LOOK_BACK, HORIZON); v_idx=~np.isnan(Y_o); X_seq_o=X_seq_o[v_idx]; Y_o=Y_o[v_idx]
Y_o = np.log1p(Y_o)
print(f"Total usable samples: {X_seq_o.shape[0]}")

# --- 6. CREATE DATA SPLITS ---
sp_idx=int(X_seq_o.shape[0]*.8)
X_tr_v_o, X_ts_o = X_seq_o[:sp_idx], X_seq_o[sp_idx:]
Y_tr_v_r, Y_ts_r = Y_o[:sp_idx], Y_o[sp_idx:]
Y_tr_v_p, Y_ts_p = (np.expm1(Y_tr_v_r)>0).astype(int), (np.expm1(Y_ts_r)>0).astype(int)

# Scale sequential features
sc=StandardScaler()
X_tr_v_f_s=sc.fit_transform(np.nan_to_num(X_tr_v_o.reshape(X_tr_v_o.shape[0],-1)))
X_tr_v_s=X_tr_v_f_s.reshape(X_tr_v_o.shape).astype(np.float32)

# Create final Train/Val split for tuning
(X_tr_s, X_v_s,
 Y_tr_r, Y_v_r,
 Y_tr_p, Y_v_p) = train_test_split(
     X_tr_v_s, Y_tr_v_r, Y_tr_v_p, 
     test_size=0.2, random_state=GLOBAL_SEED
)

# Create smaller subset for Keras tuning
print(f"Creating tuning subset ({TUNING_SUBSET_FRACTION*100}%) for Keras...")
subset_size = int(len(X_tr_s) * TUNING_SUBSET_FRACTION)
indices = np.random.choice(len(X_tr_s), subset_size, replace=False)

X_tr_s_tune = X_tr_s[indices]
Y_tr_r_tune = Y_tr_r[indices]
Y_tr_p_tune = Y_tr_p[indices]
Y_tr_targets_tune = {'prob_output': Y_tr_p_tune, 'reg_output': Y_tr_r_tune}
Y_v_targets = {'prob_output': Y_v_p, 'reg_output': Y_v_r}

print(f"  Train (Seq): {X_tr_s.shape} | Val (Seq): {X_v_s.shape}")
print(f"  Keras Tune (Seq): {X_tr_s_tune.shape}")


# --- 7. TUNE L0 KERAS MODELS ---
print("\n  [GPU] Keras Model Tuning...")
es_callback = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# --- TCN-BiGRU Tuning ---
def obj_tcn(trial):
    tf.keras.backend.clear_session()
    params = {
        'conv_filters': trial.suggest_categorical('conv_filters', [64, 128, 192]),
        'gru_units': trial.suggest_categorical('gru_units', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.4),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    
    # Removed strategy.scope()
    model = create_tcn_bigru_model(LOOK_BACK, N_FEATS, params)
    model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100, 
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # Return only the first value (total_loss)
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_tcn = optuna.create_study(direction='minimize', 
                                sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                                pruner=optuna.pruners.MedianPruner())
study_tcn.optimize(obj_tcn, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_tcn = study_tcn.best_params


# --- BiLSTM Tuning ---
def obj_lstm(trial):
    tf.keras.backend.clear_session()
    params = {
        'lstm_units_1': trial.suggest_categorical('lstm_units_1', [96, 128, 192]),
        'lstm_units_2': trial.suggest_categorical('lstm_units_2', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    
    # Removed strategy.scope()
    model = create_bilstm_model(LOOK_BACK, N_FEATS, params)
    model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100,
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # Return only the first value (total_loss)
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_lstm = optuna.create_study(direction='minimize', 
                                 sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                                 pruner=optuna.pruners.MedianPruner())
study_lstm.optimize(obj_lstm, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_lstm = study_lstm.best_params


# --- 8. PRINTABLE RESULTS ---
print("\n" + "="*50)
print("     KERAS TUNING COMPLETE - BEST PARAMETERS")
print("="*50 + "\n")

print(f"  Best TCN-BiGRU (Val Loss: {study_tcn.best_value:.4f})")
print(f"bp_tcn_global = {bp_tcn}")

print("\n" + "-"*50 + "\n")

print(f"  Best BiLSTM (Val Loss: {study_lstm.best_value:.4f})")
print(f"bp_lstm_global = {bp_lstm}")

print(f"\n" + "="*50)
print("Copy the dictionaries above and paste them")
print("into your main script (Section 4).")
print("="*50)

In [ ]:

# --- 0. IMPORTS ---
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GRU, LSTM, Dropout, Conv1D, Input, Bidirectional, Layer
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import optuna
from optuna.integration import TFKerasPruningCallback
import warnings
import os

# --- 1. SCRIPT-WIDE SETTINGS ---
GLOBAL_SEED = 42
N_TRIALS_KERAS = 15 # <-- Set how many trials to run (e.g., 15-25)
TUNING_SUBSET_FRACTION = 0.75 # Use 75% of data for tuning (faster)

# Suppress warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = str(GLOBAL_SEED)

# Set seeds
np.random.seed(GLOBAL_SEED)
tf.keras.utils.set_random_seed(GLOBAL_SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

# --- 2. SETUP STRATEGY (SINGLE GPU / CPU) ---
print("Running on single device (GPU/CPU). Multi-GPU strategy removed.")
# Set a fixed batch size. 128 is a good default.
GLOBAL_BATCH_SIZE = 128 
print(f'Global Batch Size: {GLOBAL_BATCH_SIZE}')


# --- 3. CUSTOM LOSS & ATTENTION LAYER ---
class Attention(Layer):
    def __init__(self,**kwargs): super(Attention,self).__init__(**kwargs)
    def build(self, input_shape):
        self.W=self.add_weight(name='att_W',shape=(input_shape[-1],1),initializer='glorot_uniform',trainable=True)
        self.b=self.add_weight(name='att_b',shape=(input_shape[1],1),initializer='zeros',trainable=True); super(Attention,self).build(input_shape)
    def call(self, x): e=tf.tanh(tf.tensordot(x,self.W,axes=1)+self.b); a=tf.nn.softmax(e,axis=1); o=x*a; return tf.reduce_sum(o,axis=1)

# --- 4. MODEL DEFINITIONS (MODIFIED FOR OPTUNA) ---
def create_tcn_bigru_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    
    # ---
    # --- FIX: Changed 'causal' to 'same' to avoid CUDNN bug ---
    # ---
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=1, kernel_regularizer=reg)(i)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=2, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    t=Conv1D(params['conv_filters'], 3, activation='relu', padding='same', dilation_rate=4, kernel_regularizer=reg)(t)
    t=Dropout(params['dropout_rate'])(t)
    # --- End of Fix ---

    bg=Bidirectional(GRU(params['gru_units'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(t)
    att=Attention()(bg)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def create_bilstm_model(n_timesteps, n_features, params):
    reg = l2(params.get('l2_reg', 1e-4))
    i=Input(shape=(n_timesteps,n_features))
    l=Bidirectional(LSTM(params['lstm_units_1'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(i)
    l=Dropout(params['dropout_rate'])(l)
    l=Bidirectional(LSTM(params['lstm_units_2'], activation='tanh', return_sequences=True, kernel_regularizer=reg))(l)
    l=Dropout(params['dropout_rate'])(l)
    att=Attention()(l)
    p=Dense(1,activation='sigmoid',name='prob_output')(att)
    r=Dense(1,activation='linear',name='reg_output',dtype='float32')(att)
    return Model(inputs=i,outputs=[p,r])

def compile_keras_model(model, learning_rate, prob_weight=0.2, reg_weight=1.8):
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss={'prob_output':BinaryCrossentropy(), 'reg_output':'mean_squared_error'},
                  loss_weights={'prob_output': prob_weight,'reg_output': reg_weight})
    return model

# --- 5. DATA PREPARATION (GLOBAL) ---
DATASET_NAME = "lengpui" 
try:
    df=pd.read_csv(f'{DATASET_NAME}.csv')
    print(f"  Successfully loaded '{DATASET_NAME}.csv'")
except FileNotFoundError:
    print(f"  Error: '{DATASET_NAME}.csv' not found. Trying fallback...")
    try:
        df=pd.read_csv(f'/kaggle/input/dataset2/{DATASET_NAME}.csv')
        print(f"  Successfully loaded from fallback path.")
    except FileNotFoundError:
        print(f"  Error: Fallback path also failed. Exiting.")
        exit()

# Pre-processing
df['Date']=pd.to_datetime({'year':df['YEAR'],'month':df['MN'],'day':df['DT']}); df.set_index('Date',inplace=True)
num_cols=['RF','MSLP','DBT','RH','ONI','DMI']; [df[c].apply(pd.to_numeric,errors='coerce') for c in num_cols]
daily=df.groupby(df.index.date).agg({c:('sum' if c=='RF' else 'mean') for c in num_cols}); daily.index=pd.to_datetime(daily.index); daily.dropna(subset=num_cols,inplace=True)
daily['RF_SSA']=daily['RF'].rolling(7,center=True).mean().fillna(daily['RF']); daily['RF_7m']=daily['RF'].rolling(7).mean().shift(1); daily['RF_7s']=daily['RF'].rolling(7).std().shift(1); daily['sin_d']=np.sin(2*np.pi*daily.index.dayofyear/365.25); daily['cos_d']=np.cos(2*np.pi*daily.index.dayofyear/365.25)
daily['RF_l1']=daily['RF'].shift(1); daily['DBT_l1']=daily['DBT'].shift(1); daily['RH_MSLP_I']=(daily['RH']/daily['MSLP']).replace([np.inf,-np.inf],np.nan); daily.replace([np.inf,-np.inf],np.nan,inplace=True); daily.dropna(inplace=True); daily['RF']=pd.to_numeric(daily['RF'],errors='coerce').fillna(0)
HORIZON=3; LOOK_BACK=7; FEATURES=['RF_SSA','MSLP','DBT','RH','RF_7m','RF_7s','sin_d','cos_d','ONI','DMI','RF_l1','DBT_l1','RH_MSLP_I']; FEATURES=[f for f in FEATURES if f in daily.columns]; N_FEATS=len(FEATURES); print(f"Features ({N_FEATS}): {FEATURES}")

def create_ts_feats(data,feats,target,lb,hz):
    X,Y=[],[]; ts=data[target].shift(-hz); ts=pd.to_numeric(ts,errors='coerce'); vfd=data[feats].dropna(); max_idx=len(vfd)-lb-hz+1; [ (X.append(fw),Y.append(ts.iloc[i+lb-1])) for i in range(max_idx) if (i+lb-1)<len(ts) and not pd.isna(ts.iloc[i+lb-1]) and (fw:=vfd.iloc[i:i+lb].values).shape[0]==lb ]; X=np.array(X).astype(np.float32); Y=np.array(Y).astype(np.float32); print(f"X:{X.shape}, Y:{Y.shape}"); return X,Y

X_seq_o, Y_o = create_ts_feats(daily, FEATURES, 'RF', LOOK_BACK, HORIZON); v_idx=~np.isnan(Y_o); X_seq_o=X_seq_o[v_idx]; Y_o=Y_o[v_idx]
Y_o = np.log1p(Y_o)
print(f"Total usable samples: {X_seq_o.shape[0]}")

# --- 6. CREATE DATA SPLITS ---
sp_idx=int(X_seq_o.shape[0]*.8)
X_tr_v_o, X_ts_o = X_seq_o[:sp_idx], X_seq_o[sp_idx:]
Y_tr_v_r, Y_ts_r = Y_o[:sp_idx], Y_o[sp_idx:]
Y_tr_v_p, Y_ts_p = (np.expm1(Y_tr_v_r)>0).astype(int), (np.expm1(Y_ts_r)>0).astype(int)

# Scale sequential features
sc=StandardScaler()
X_tr_v_f_s=sc.fit_transform(np.nan_to_num(X_tr_v_o.reshape(X_tr_v_o.shape[0],-1)))
X_tr_v_s=X_tr_v_f_s.reshape(X_tr_v_o.shape).astype(np.float32)

# Create final Train/Val split for tuning
(X_tr_s, X_v_s,
 Y_tr_r, Y_v_r,
 Y_tr_p, Y_v_p) = train_test_split(
     X_tr_v_s, Y_tr_v_r, Y_tr_v_p, 
     test_size=0.2, random_state=GLOBAL_SEED
)

# Create smaller subset for Keras tuning
print(f"Creating tuning subset ({TUNING_SUBSET_FRACTION*100}%) for Keras...")
subset_size = int(len(X_tr_s) * TUNING_SUBSET_FRACTION)
indices = np.random.choice(len(X_tr_s), subset_size, replace=False)

X_tr_s_tune = X_tr_s[indices]
Y_tr_r_tune = Y_tr_r[indices]
Y_tr_p_tune = Y_tr_p[indices]
Y_tr_targets_tune = {'prob_output': Y_tr_p_tune, 'reg_output': Y_tr_r_tune}
Y_v_targets = {'prob_output': Y_v_p, 'reg_output': Y_v_r}

print(f"  Train (Seq): {X_tr_s.shape} | Val (Seq): {X_v_s.shape}")
print(f"  Keras Tune (Seq): {X_tr_s_tune.shape}")


# --- 7. TUNE L0 KERAS MODELS ---
print("\n  [GPU] Keras Model Tuning...")
es_callback = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# --- TCN-BiGRU Tuning ---
def obj_tcn(trial):
    tf.keras.backend.clear_session()
    params = {
        'conv_filters': trial.suggest_categorical('conv_filters', [64, 128, 192]),
        'gru_units': trial.suggest_categorical('gru_units', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.4),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    
    # Removed strategy.scope()
    model = create_tcn_bigru_model(LOOK_BACK, N_FEATS, params)
    model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100, 
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # Return only the first value (total_loss)
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_tcn = optuna.create_study(direction='minimize', 
                                sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                                pruner=optuna.pruners.MedianPruner())
study_tcn.optimize(obj_tcn, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_tcn = study_tcn.best_params


# --- BiLSTM Tuning ---
def obj_lstm(trial):
    tf.keras.backend.clear_session()
    params = {
        'lstm_units_1': trial.suggest_categorical('lstm_units_1', [96, 128, 192]),
        'lstm_units_2': trial.suggest_categorical('lstm_units_2', [96, 128, 192]),
        'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True),
        'l2_reg': trial.suggest_float('l2_reg', 1e-5, 1e-3, log=True)
    }
    
    # Removed strategy.scope()
    model = create_bilstm_model(LOOK_BACK, N_FEATS, params)
    model = compile_keras_model(model, params['learning_rate'])
    
    pruning_callback = TFKerasPruningCallback(trial, "val_loss")
    model.fit(X_tr_s_tune, Y_tr_targets_tune, # <-- Use tuning subset
              validation_data=(X_v_s, Y_v_targets),
              epochs=100,
              batch_size=GLOBAL_BATCH_SIZE,
              callbacks=[es_callback, pruning_callback],
              verbose=0)
    
    # Return only the first value (total_loss)
    return model.evaluate(X_v_s, Y_v_targets, batch_size=GLOBAL_BATCH_SIZE, verbose=0)[0]

study_lstm = optuna.create_study(direction='minimize', 
                                 sampler=optuna.samplers.TPESampler(seed=GLOBAL_SEED),
                                 pruner=optuna.pruners.MedianPruner())
study_lstm.optimize(obj_lstm, n_trials=N_TRIALS_KERAS, show_progress_bar=True)
bp_lstm = study_lstm.best_params


# --- 8. PRINTABLE RESULTS ---
print("\n" + "="*50)
print("     KERAS TUNING COMPLETE - BEST PARAMETERS")
print("="*50 + "\n")

print(f"  Best TCN-BiGRU (Val Loss: {study_tcn.best_value:.4f})")
print(f"bp_tcn_global = {bp_tcn}")

print("\n" + "-"*50 + "\n")

print(f"  Best BiLSTM (Val Loss: {study_lstm.best_value:.4f})")
print(f"bp_lstm_global = {bp_lstm}")

print(f"\n" + "="*50)
print("Copy the dictionaries above and paste them")
print("into your main script (Section 4).")
print("="*50)